## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Phone type and paying for app features: tau

**Goodman-Kruskal's Tau** · a worked example from the StatNex teaching guide

**The story.** An app developer asked 285 smartphone users whether they would pay for premium app features. It also recorded whether each person owns a premium or a budget phone.

**The question.** If we know someone's phone, how much better can we guess their answer, and the other way round?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/goodman_kruskal_tau.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Phone owners,Agree,Neutral,Disagree
Premium-phone owners,110.0,30.0,25.0
Budget-phone owners,28.0,22.0,70.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of counts: one variable in the rows, the other in the columns. No order is
# needed on either.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Agree", "Neutral", "Disagree"]
table = data.set_index("Phone owners")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
# is_total: defined in the helper cell totals above (totals.py in the code download)

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## Goodman and Kruskal's tau

In [ ]:
# How much less often you would guess a case's category wrong if you knew the other
# variable. The error of a guess is measured by the qualitative variance (1 - sum p^2) / 2
# (half the Gini index). Without the other variable it is the variance of the marginal
# shares; with it, the average of the conditional variances weighted by group size.
# tau = (overall - weighted conditional) / overall. Neither SciPy nor statsmodels has it.
def qualitative_variance(shares):
    return (1 - (shares**2).sum()) / 2


total = table.to_numpy().sum()


# Each column of `groups` is one group; its rows are the categories being predicted.
def goodman_kruskal_tau(groups):
    conditional = groups.apply(lambda counts: qualitative_variance(counts / counts.sum()))
    overall = qualitative_variance(groups.sum(axis=1) / total)
    weighted = (groups.sum() / total * conditional).sum()
    variance = pd.concat([conditional, pd.Series({"Overall": overall})])
    return variance, (overall - weighted) / overall


# Knowing the row: the column is the variable predicted.
variance_rows, tau_column = goodman_kruskal_tau(table.T)
print("\nQualitative variance, within each row:")
print(variance_rows.to_string())
print(f"tau (column predicted from the row) = {tau_column:.4f}")

# Knowing the column: the row is the variable predicted.
variance_columns, tau_row = goodman_kruskal_tau(table)
print("\nQualitative variance, within each column:")
print(variance_columns.to_string())
print(f"tau (row predicted from the column) = {tau_row:.4f}")

## What the result says

Goodman-Kruskal's tau says how much knowing one variable reduces the errors in guessing the other. It runs from 0 (no help) to 1 (perfect guess), and it has two values, one for each direction.

With the answer as the dependent variable, tau is 0.147: knowing the phone removes about 14.7% of the errors in guessing the answer. With the phone as the dependent variable, tau is 0.231, so the answer says more about the phone. The table shows why: 110 of 165 premium-phone owners agree, against 28 of 120 budget-phone owners.

So the phone helps a little in guessing who would pay, and the answer helps more in guessing the phone.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.